In [1]:
print("Shivbaba")

Shivbaba


In [ ]:
import os
import gradio as gr
from pypdf import PdfReader
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

%pip install -q gradio pypdf openai scikit-learn



def get_client():
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("Set the OPENAI_API_KEY environment variable to use the assistant.")
    return OpenAI()


def extract_resume_chunks(files, chunk_size=1200, overlap=200):
    chunks = []
    for file_path in files or []:
        reader = PdfReader(file_path)
        filename = os.path.basename(file_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            start = 0
            while start < len(text):
                excerpt = text[start:start + chunk_size].strip()
                if excerpt:
                    chunks.append({
                        "filename": filename,
                        "page": page_number,
                        "text": excerpt,
                    })
                start += chunk_size - overlap

    return chunks


def retrieve(query, chunks, limit=8):
    if not chunks:
        return []

    texts = [chunk["text"] for chunk in chunks]
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(texts + [query])
    similarities = cosine_similarity(matrix[-1], matrix[:-1]).ravel()

    indices = similarities.argsort()[::-1][:limit]
    return [chunks[i] for i in indices if similarities[i] > 0]


def format_context(chunks):
    return "\n\n".join(
        f"[{item['filename']}, page {item['page']}]\n{item['text']}"
        for item in chunks
    )


def evaluate_resumes(files, job_description):
    if not files:
        return "Upload at least one PDF resume."
    if not job_description.strip():
        return "Enter a job description."

    try:
        client = get_client()
        results = []

        for file_path in files:
            filename = os.path.basename(file_path)
            resume_chunks = [
                chunk for chunk in extract_resume_chunks([file_path])
            ]
            context = format_context(retrieve(job_description, resume_chunks, limit=10))

            if not context:
                results.append(f"## {filename}\nNo readable resume text was found.")
                continue

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=0,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Evaluate the resume against the job description. "
                            "Use the job description only as evaluation criteria. "
                            "Every factual claim about the candidate must be supported "
                            "by the resume excerpts and cited as [filename, page N]. "
                            "Do not infer missing details; say 'Not evidenced in the resume'. "
                            "Provide an evidence-based fit score from 0–100, relevant "
                            "strengths, gaps or unverified requirements, and a brief summary."
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Job description:\n{job_description[:10000]}\n\n"
                            f"Resume excerpts:\n{context}"
                        ),
                    },
                ],
            )
            results.append(f"## {filename}\n{response.choices[0].message.content}")

        return "\n\n".join(results)

    except Exception as exc:
        return f"Could not evaluate resumes: {exc}"


def answer_question(question, history, chunks):
    history = history or []
    if not question.strip():
        return history, ""

    if not chunks:
        answer = "Upload one or more resumes first."
    else:
        try:
            context = format_context(retrieve(question, chunks, limit=8))
            if not context:
                answer = "The uploaded resumes do not provide evidence to answer that."
            else:
                response = get_client().chat.completions.create(
                    model="gpt-4o-mini",
                    temperature=0,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "Answer only using the supplied resume excerpts. "
                                "Do not use the job description, outside knowledge, or "
                                "unsupported assumptions. Cite factual claims as "
                                "[filename, page N]. If the excerpts do not contain the "
                                "answer, say so."
                            ),
                        },
                        {
                            "role": "user",
                            "content": f"Resume excerpts:\n{context}\n\nQuestion: {question}",
                        },
                    ],
                )
                answer = response.choices[0].message.content
        except Exception as exc:
            answer = f"Could not answer the question: {exc}"

    return history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ], ""


with gr.Blocks() as app:
    gr.Markdown("# Resume RAG Assistant")
    gr.Markdown("Upload PDF resumes, enter a job description, and evaluate candidates.")

    resume_files = gr.File(
        label="Resume PDFs",
        file_count="multiple",
        file_types=[".pdf"],
        type="filepath",
    )
    job_description = gr.Textbox(
        label="Job Description",
        lines=8,
        placeholder="Paste the job description here...",
    )
    resume_state = gr.State([])

    resume_files.change(
        fn=lambda files: (
            extract_resume_chunks(files),
            f"Loaded {len(files or [])} PDF(s).",
        ),
        inputs=resume_files,
        outputs=[resume_state, gr.Markdown()],
    )

    evaluate_button = gr.Button("Evaluate resumes")
    evaluation_output = gr.Markdown()
    evaluate_button.click(
        fn=evaluate_resumes,
        inputs=[resume_files, job_description],
        outputs=evaluation_output,
    )

    gr.Markdown("## Ask about the uploaded resumes")
    chatbot = gr.Chatbot(type="messages")
    question = gr.Textbox(label="Question", placeholder="Ask a question about the resumes...")
    question.submit(
        fn=answer_question,
        inputs=[question, chatbot, resume_state],
        outputs=[chatbot, question],
    )

app.launch()